In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Función de Circuit de IBM

> **Note:** * Las Qiskit Functions son una función experimental disponible únicamente para usuarios del Plan Premium, Plan Flex y Plan On-Prem (a través de la API de IBM Quantum Platform) de IBM Quantum&reg;. Están en estado de versión preliminar y pueden cambiar.

## Descripción general
La función de Circuit de IBM&reg; recibe [PUBs abstractos](./primitive-input-output) como entradas y devuelve valores de expectación mitigados como salidas. Esta función de Circuit incluye un pipeline automatizado y personalizado para que los investigadores puedan centrarse en el descubrimiento de algoritmos y aplicaciones.

## Descripción
Después de enviar tu PUB, tus circuitos abstractos y observables se transpilan automáticamente, se ejecutan en hardware y se post-procesan para devolver valores de expectación mitigados. Para ello, combina las siguientes herramientas:

- [Qiskit Transpiler Service](./qiskit-transpiler-service), incluida la selección automática de pases de transpilación heurísticos e impulsados por IA para traducir tus circuits abstractos a circuits ISA optimizados para el hardware
- [Supresión y mitigación de errores necesarias para computación a escala utilitaria](./error-mitigation-and-suppression-techniques), incluyendo twirling de medición y de Gate, desacoplamiento dinámico, Twirled Readout Error eXtinction (TREX), Zero-Noise Extrapolation (ZNE) y Probabilistic Error Amplification (PEA)
- [Qiskit Runtime Estimator](./get-started-with-primitives), para ejecutar PUBs ISA en hardware y devolver valores de expectación mitigados

![IBM Circuit function](../docs/images/guides/ibm-circuit-function/ibm-circuit-function.svg)

## Primeros pasos
Autentícate con tu [clave de API](http://quantum.cloud.ibm.com/) y selecciona la Qiskit Function de la siguiente manera. (Este fragmento asume que ya has [guardado tu cuenta](/guides/functions#install-qiskit-functions-catalog-client) en tu entorno local.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("ibm/circuit-function")

## Ejemplo
Para comenzar, prueba este ejemplo básico:

In [2]:
from qiskit.circuit.random import random_circuit
from qiskit_ibm_runtime import QiskitRuntimeService

# You can skip this step if you have a target backend, e.g.
# backend_name = "ibm_brisbane"
# You'll need to specify the credentials when initializing QiskitRuntimeService, if they were not previously saved.
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit = random_circuit(num_qubits=2, depth=2, seed=42)
observable = "Z" * circuit.num_qubits
pubs = [(circuit, observable)]

job = function.run(
    # Use `backend_name=backend_name` if you didn't initialize a backend object
    backend_name=backend.name,
    pubs=pubs,
)

Consulta el [estado](/guides/functions#check-job-status) de la carga de trabajo de tu Qiskit Function o recupera los [resultados](/guides/functions#retrieve-results) de la siguiente manera:

In [3]:
print(job.status())
result = job.result()

QUEUED


The results have the same format as an [Estimator result](/docs/guides/primitive-input-output#estimator-output):

In [4]:
print(f"The result of the submitted job had {len(result)} PUB\n")
print(
    f"The associated PubResult of this job has the following DataBins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB

The associated PubResult of this job has the following DataBins:
 DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>))

And this DataBin has attributes: dict_keys(['evs', 'stds', 'ensemble_standard_error'])
The expectation values measured from this PUB are: 
1.02116704805492


Los resultados tienen el mismo formato que un [resultado de Estimator](/guides/primitive-input-output#estimator-output):

In [5]:
options = {"mitigation_level": 2}

job = function.run(backend_name=backend.name, pubs=pubs, options=options)

#### All available options

In addition to `mitigation_level`, the IBM Circuit function also provides a select number of advanced options that allow you to fine-tune the cost/accuracy trade-off. The following table shows all the available options:

| Option | Sub-option | Sub-sub-option | Description | Choices | Default |
|-|-|-|-|-|-|
| default_precision |  |  | The default precision to use for any PUB or `run()`<br/>call that does not specify one.<br/>Each input PUB can specify its own precision. If the `run()` method is given a precision, then that value is used for all PUBs in the `run()` call that do not specify their own.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Maximum execution time in seconds, which is based<br/>on QPU usage (not wall clock time). QPU usage is the<br/>amount of time that the QPU is dedicated to processing your job. If a job exceeds this time limit, it is forcibly canceled. | Integer number of seconds in the range [1, 10800] |  |
| mitigation_level |  |  | How much error suppression and mitigation to apply. Refer to the [Mitigation level](#mitigation-level) section for more information about the methods used at each level. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | How much optimization to perform on the circuits. [Higher levels](/docs/guides/set-optimization) generate more optimized circuits, at the expense of longer transpilation time. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Whether to enable dynamical decoupling. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) for the explanation of the method.  | True/False | True |
|  | sequence_type |  | Which dynamical decoupling sequence to use.<br/>* `XX`: use the sequence `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: use the sequence `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: use the sequence<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Whether to apply 2-qubit Clifford gate twirling. | True/False | False |
|  | enable_measure |  | Whether to enable twirling of measurements. | True/False | True |
| resilience | measure_mitigation |  | Whether to enable TREX measurement error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) for the explanation of the method.  | True/False | True |
|  | zne_mitigation |  | Whether to turn on Zero Noise Extrapolation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) for the explanation of the method.  | True/False | False |
|  | zne | amplifier | Which technique to use for amplifying noise. One of: <br/> - `gate_folding` (default) uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are chosen randomly.<br/><br/> - `gate_folding_front` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the front of the topologically ordered DAG circuit.<br/><br/> - `gate_folding_back` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the back of the topologically ordered DAG circuit.<br/><br/> - `pea` uses a technique called Probabilistic error amplification (PEA) to amplify noise. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) for the explanation of the method.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Noise factors to use for noise amplification. | list of floats; each float >= 1 | (1, 1.5, 2) for PEA, and (1, 3, 5) otherwise. |
|  |  | extrapolator | Noise factors to evaluate the fit extrapolation models at. This option does not affect execution or model fitting in any way; it only determines the points at which the `extrapolator` objects are evaluated to be returned in the data fields called `evs_extrapolated` and `stds_extrapolated`. | one or more of `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Whether to turn on Probabilistic Error Cancellation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) for the explanation of the method.  | True/False | False |
|  | pec | max_overhead | The maximum circuit sampling overhead allowed, or `None` for no maximum. | None/ integer >1 | 100 |

In the following example, setting the mitigation level to 1 initially turns off ZNE mitigation, but setting `zne_mitigation` to `True` overrides the relevant setup from `mitigation_level`.

In [6]:
options = {"mitigation_level": 1, "resilience": {"zne_mitigation": True}}

## Entradas
Consulta la siguiente tabla para ver todos los parámetros de entrada que acepta la función de Circuit de IBM. La sección [Opciones](#options) que sigue detalla más las `options` disponibles.

| Nombre      | Tipo                       | Descripción                                                                                                                                                                                                                         | Obligatorio | Valor por defecto                                                                  | Ejemplo                                  |
|-----------|----------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|------------------------------------------|
| backend_name   | str                        | Nombre del backend sobre el que realizar la consulta.                                                                                                                                                                                              | Sí      | N/A                                                                      | `ibm_fez`                                |
| pubs      | Iterable[EstimatorPubLike] | Un iterable de objetos abstractos tipo PUB (primitive unified bloc), como tuplas `(circuit, observables)` o `(circuit, observables, parameter_values)`. Consulta [Descripción general de PUBs](/guides/primitive-input-output#overview-of-pubs) para más información. Los circuits pueden ser abstractos (no ISA). | Sí      | N/A                                                                      | (circuit, observables, parameter_values) |
| options   | dict                       | Opciones de entrada. Consulta la sección **Opciones** para más detalles.                                                                                                                                                                                | No       | Consulta la sección **Opciones** para más detalles.                                                   | `{"optimization_level": 3}`                |
| instance  | str                        | El nombre del recurso en la nube de la instancia a utilizar en ese formato.                                                                                                                                                                                        | No       | Se elige una aleatoriamente si tu cuenta tiene acceso a varias instancias. | `CRN`                   |

### Opciones
#### Estructura
Al igual que con los primitivos de Qiskit Runtime, las opciones de la función de Circuit de IBM se pueden especificar como un diccionario anidado. Las opciones de uso habitual, como ``optimization_level`` y ``mitigation_level``, se encuentran en el primer nivel. Otras opciones más avanzadas se agrupan en diferentes categorías, como ``resilience``.

#### Valores por defecto
Si no especificas un valor para una opción, se usa el valor por defecto definido por el servicio.

#### Nivel de mitigación
La función de Circuit de IBM también admite `mitigation_level`. El nivel de mitigación especifica cuánta supresión y mitigación de errores aplicar al trabajo. Los niveles más altos generan resultados más precisos, a costa de tiempos de procesamiento más largos. El grado de reducción de errores depende del método aplicado. El nivel de mitigación abstrae la elección detallada de métodos de mitigación y supresión de errores para que los usuarios puedan razonar sobre la compensación entre costo y precisión adecuada a su aplicación. La siguiente tabla muestra los métodos correspondientes para cada nivel.

> **Note:** Aunque los nombres son similares, `mitigation_level` aplica técnicas distintas a las que usa `resilience_level` de Estimator.

Al igual que ``resilience_level`` en Qiskit Runtime Estimator, ``mitigation_level`` especifica un conjunto base de opciones seleccionadas. Cualquier opción que especifiques manualmente además del nivel de mitigación se aplica encima del conjunto base de opciones definido por el nivel de mitigación. Por lo tanto, en principio, podrías establecer el nivel de mitigación en 1 pero luego desactivar la mitigación de medición, aunque esto no se recomienda.

| **Nivel de mitigación** | **Técnica** |
|:-:|:-:|
| 1 [Predeterminado] | Desacoplamiento dinámico + twirling de medición + TREX  |
| 2 | Nivel 1 + twirling de Gate + ZNE mediante gate folding |
| 3 | Nivel 1 + twirling de Gate + ZNE mediante PEA |

El siguiente ejemplo muestra cómo establecer el nivel de mitigación:

In [7]:
print(f"The result of the submitted job had {len(result)} PUB")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)
print(f"And the associated metadata is: \n{result[0].metadata}")

The result of the submitted job had 1 PUB
The expectation values measured from this PUB are: 
1.02116704805492
And the associated metadata is: 
{'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}


#### Todas las opciones disponibles
Además de `mitigation_level`, la función de Circuit de IBM también ofrece un número selecto de opciones avanzadas que te permiten ajustar con precisión la compensación entre costo y precisión. La siguiente tabla muestra todas las opciones disponibles:

| Opción | Sub-opción | Sub-sub-opción | Descripción | Valores posibles | Valor por defecto |
|-|-|-|-|-|-|
| default_precision |  |  | La precisión predeterminada a usar para cualquier PUB o llamada a `run()`<br/>que no especifique una.<br/>Cada PUB de entrada puede especificar su propia precisión. Si al método `run()` se le da una precisión, ese valor se usa para todos los PUBs de la llamada a `run()` que no especifiquen la propia.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Tiempo máximo de ejecución en segundos, basado<br/>en el uso de QPU (no en tiempo de reloj). El uso de QPU es el<br/>tiempo durante el cual la QPU está dedicada a procesar tu trabajo. Si un trabajo supera este límite de tiempo, se cancela de forma forzosa. | Número entero de segundos en el rango [1, 10800] |  |
| mitigation_level |  |  | Cuánta supresión y mitigación de errores aplicar. Consulta la sección [Nivel de mitigación](#mitigation-level) para más información sobre los métodos usados en cada nivel. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | Cuánta optimización realizar en los circuits. Los [niveles más altos](/guides/set-optimization) generan circuits más optimizados, a costa de un mayor tiempo de transpilación. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Si se habilita el desacoplamiento dinámico. Consulta [Técnicas de supresión y mitigación de errores](/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) para la explicación del método.  | True/False | True |
|  | sequence_type |  | Qué secuencia de desacoplamiento dinámico usar.<br/>* `XX`: usa la secuencia `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: usa la secuencia `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: usa la secuencia<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Si se aplica twirling de Gate Clifford de 2 Qubits. | True/False | False |
|  | enable_measure |  | Si se habilita el twirling de mediciones. | True/False | True |
| resilience | measure_mitigation |  | Si se habilita el método de mitigación de errores de medición TREX. Consulta [Técnicas de supresión y mitigación de errores](/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) para la explicación del método.  | True/False | True |
|  | zne_mitigation |  | Si se activa el método de mitigación de errores Zero Noise Extrapolation. Consulta [Técnicas de supresión y mitigación de errores](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) para la explicación del método.  | True/False | False |
|  | zne | amplifier | Qué técnica usar para amplificar el ruido. Una de: <br/> - `gate_folding` (predeterminada) usa gate folding de 2 Qubits para amplificar el ruido. Si el factor de ruido requiere amplificar solo un subconjunto de los Gates, estos se eligen aleatoriamente.<br/><br/> - `gate_folding_front` usa gate folding de 2 Qubits para amplificar el ruido. Si el factor de ruido requiere amplificar solo un subconjunto de los Gates, estos se seleccionan desde el frente del circuit DAG ordenado topológicamente.<br/><br/> - `gate_folding_back` usa gate folding de 2 Qubits para amplificar el ruido. Si el factor de ruido requiere amplificar solo un subconjunto de los Gates, estos se seleccionan desde el final del circuit DAG ordenado topológicamente.<br/><br/> - `pea` usa una técnica llamada Probabilistic error amplification (PEA) para amplificar el ruido. Consulta [Técnicas de supresión y mitigación de errores](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) para la explicación del método.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Factores de ruido a usar para la amplificación del ruido. | lista de flotantes; cada flotante >= 1 | (1, 1.5, 2) para PEA, y (1, 3, 5) en otros casos. |
|  |  | extrapolator | Factores de ruido en los que evaluar los modelos de extrapolación ajustados. Esta opción no afecta la ejecución ni el ajuste del modelo de ninguna manera; solo determina los puntos en los que los objetos `extrapolator` se evalúan para devolverse en los campos de datos llamados `evs_extrapolated` y `stds_extrapolated`. | uno o más de `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Si se activa el método de mitigación de errores Probabilistic Error Cancellation. Consulta [Técnicas de supresión y mitigación de errores](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) para la explicación del método.  | True/False | False |
|  | pec | max_overhead | La sobrecarga máxima permitida de muestreo del circuit, o `None` para no establecer un máximo. | None/ entero >1 | 100 |

En el siguiente ejemplo, establecer el nivel de mitigación en 1 desactiva inicialmente la mitigación ZNE, pero establecer `zne_mitigation` en `True` sobreescribe la configuración relevante de `mitigation_level`.

In [8]:
job = function.run(
    backend_name="bad_backend_name", pubs=pubs, options=options
)

print(job.result())

QiskitServerlessException: "Traceback (most recent call last):\n  File \"/runner/runner.py\", line 10, in run\n    func = CircuitFunction(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/runner/circuit_function/circuit_function.py\", line 87, in __init__\n    self._backend = self._service.backend(\n                    ^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 754, in backend\n    backends = self.backends(name, instance=instance, use_fractional_gates=use_fractional_gates)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 497, in backends\n    raise QiskitBackendNotFoundError(\"No backend matches the criteria.\")\nqiskit.providers.exceptions.QiskitBackendNotFoundError: 'No backend matches the criteria.'\n"

## Salidas
La salida de una función de Circuit es un [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), que contiene dos campos:

- Uno o más objetos [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult). Estos se pueden indexar directamente desde el `PrimitiveResult`.

- Metadatos a nivel de trabajo.

Cada `PubResult` contiene un campo `data` y un campo `metadata`.

- El campo `data` contiene al menos un array de valores de expectación (`PubResult.data.evs`) y un array de errores estándar (`PubResult.data.stds`). También puede contener más datos, según las opciones utilizadas.

- El campo `metadata` contiene metadatos a nivel de PUB (`PubResult.metadata`).

El siguiente fragmento de código describe el formato de `PrimitiveResult` (y el `PubResult` asociado).